In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/README.txt
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/hog.py
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/extractFrames.py
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/AllLabels.csv
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TestLabels.csv
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/ValidationLabels.csv
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Test.txt
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Validation.txt
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train.txt
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Validation/400023/4000231047/4000231047.avi
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Validation/400023/4000231001/4000231001.avi
/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Validation/400023/4000231049/400023104

In [19]:
!pip install torch torchvision torchaudio facenet-pytorch librosa tqdm

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("librosa").setLevel(logging.ERROR)
logging.getLogger("numba").setLevel(logging.ERROR)

In [ ]:
import os, gc, cv2, subprocess, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, librosa, pandas as pd
import warnings, logging, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from facenet_pytorch import MTCNN
from collections import Counter

In [ ]:
warnings.filterwarnings("ignore")
logging.getLogger("librosa").setLevel(logging.ERROR)
logging.getLogger("numba").setLevel(logging.ERROR)
logging.getLogger("audioread").setLevel(logging.ERROR)
os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
class Config:
    ROOT         = Path("/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE")
    DATASET_DIR  = ROOT / "DataSet"
    LABELS_DIR   = ROOT / "Labels"
    TRAIN_LABELS = LABELS_DIR / "TrainLabels.csv"
    VAL_LABELS   = LABELS_DIR / "ValidationLabels.csv"
    TEST_LABELS  = LABELS_DIR / "TestLabels.csv"
    TRAIN_DIR    = DATASET_DIR / "Train"
    VAL_DIR      = DATASET_DIR / "Validation"
    TEST_DIR     = DATASET_DIR / "Test"
    CACHE_DIR    = Path("/kaggle/working/daisee_cache")
    SAVE_PATH    = Path("/kaggle/working/best_daisee_model.pt")
    NUM_CLASSES   = 4
    TARGET_COL    = "Engagement"  
    FRAME_RATE    = 1             
    MAX_FRAMES    = 16
    FACE_SIZE     = (112, 112)
    SCENE_SIZE    = (224, 224)
    AUDIO_SR      = 16000
    N_MELS        = 64
    EMBED_DIM     = 256
    NHEAD         = 8
    NUM_LAYERS    = 2
    DROPOUT       = 0.3

    BATCH_SIZE    = 8
    EPOCHS        = 20
    LR            = 1e-4
    DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

    CACHE_WORKERS = 4    
    DL_WORKERS    = 2    
    LOG_EVERY     = 100  

cfg = Config()
cfg.CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 58)
print("  DAiSEE Multimodal Engagement Classifier")
print("  [Disk-Cache  |  ffmpeg Audio  |  Browser-Safe]")
print("=" * 58)
print(f"  Device        : {cfg.DEVICE}")
print(f"  Target label  : {cfg.TARGET_COL}  (0=none → 3=high)")
print(f"  Max frames    : {cfg.MAX_FRAMES} @ {cfg.FRAME_RATE} fps")
print(f"  Embed dim     : {cfg.EMBED_DIM}")
print(f"  Batch size    : {cfg.BATCH_SIZE}")
print(f"  Epochs        : {cfg.EPOCHS}  |  LR: {cfg.LR}")
print(f"  Cache dir     : {cfg.CACHE_DIR}")
print(f"  Log every     : {cfg.LOG_EVERY} steps")
print("=" * 58)

  DAiSEE Multimodal Engagement Classifier
  [Disk-Cache  |  ffmpeg Audio  |  Browser-Safe]
  Device        : cuda
  Target label  : Engagement  (0=none → 3=high)
  Max frames    : 16 @ 1 fps
  Embed dim     : 256
  Batch size    : 8
  Epochs        : 20  |  LR: 0.0001
  Cache dir     : /kaggle/working/daisee_cache
  Log every     : 100 steps


In [ ]:
def log_progress(done, total, t0, tag=""):
    pct     = 100 * done / max(total, 1)
    elapsed = time.time() - t0
    eta     = elapsed / max(done, 1) * (total - done)
    print(f"  [{tag}] {done:>5}/{total}  ({pct:.0f}%)"
          f"  elapsed={elapsed:.0f}s  eta={eta:.0f}s")

In [ ]:
def build_video_index(split_dir: Path) -> dict:
    index = {}
    for root, _, files in os.walk(split_dir):
        for f in files:
            if f.lower().endswith((".avi", ".mp4", ".mkv")):
                index[Path(f).stem] = Path(root) / f
    return index

def build_label_dict(csv_path: Path, split_dir: Path) -> dict:
    print(f"\n[Labels] Reading {csv_path.name} ...")
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    video_index = build_video_index(split_dir)
    print(f"[Labels] Video files found in {split_dir.name}: {len(video_index):,}")

    found, missing = {}, 0
    for _, row in df.iterrows():
        stem  = Path(str(row["ClipID"]).strip()).stem
        label = int(row[cfg.TARGET_COL])
        if stem in video_index:
            found[stem] = (label, video_index[stem])
        else:
            missing += 1

    dist = Counter(v[0] for v in found.values())
    print(f"[Labels] Matched  : {len(found):,}  |  Missing: {missing}")
    print(f"[Labels] Class dist (label:count): { {k: dist[k] for k in sorted(dist)} }")
    return found

print(f"\n[Setup] Loading MTCNN on {cfg.DEVICE} ...", end=" ", flush=True)
mtcnn = MTCNN(image_size=cfg.FACE_SIZE[0], margin=20,
              keep_all=False, device=cfg.DEVICE)
print("ready.")

_scene_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(cfg.SCENE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def _get_face_np(rgb_np: np.ndarray) -> np.ndarray:
    """Returns face crop as float32 numpy [3,H,W] in [0,1]. Blank if not found."""
    try:
        t = mtcnn(rgb_np)
        if t is not None:
            return torch.clamp((t + 1) / 2, 0, 1).numpy()
    except Exception:
        pass
    return np.zeros((3, *cfg.FACE_SIZE), dtype=np.float32)


def _load_audio_ffmpeg(video_path: Path) -> np.ndarray:
    W = cfg.MAX_FRAMES * 4
    try:
        cmd = [
            "ffmpeg", "-i", str(video_path),
            "-vn",                     
            "-acodec", "pcm_s16le",    
            "-ar", str(cfg.AUDIO_SR), 
            "-ac", "1",                
            "-f", "s16le",             
            "-loglevel", "quiet",      
            "pipe:1"
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=30)
        if result.returncode != 0 or len(result.stdout) == 0:
            raise RuntimeError("no audio")
        y = np.frombuffer(result.stdout, dtype=np.int16).astype(np.float32)
        y /= 32768.0
    except Exception:
        y = np.zeros(cfg.AUDIO_SR * 5, dtype=np.float32)   # 5s silence

    mel    = librosa.feature.melspectrogram(y=y, sr=cfg.AUDIO_SR, n_mels=cfg.N_MELS, hop_length=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    if mel_db.shape[1] < W:
        mel_db = np.pad(mel_db, ((0, 0), (0, W - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :W]

    return mel_db[np.newaxis].astype(np.float16)   # [1, N_MELS, W]



[Setup] Loading MTCNN on cuda ... ready.


In [ ]:
def _cache_clip(args: tuple):
    stem, label, video_path = args
    cache_path = cfg.CACHE_DIR / f"{stem}.npz"
    if cache_path.exists():
        return "skip"

    T = cfg.MAX_FRAMES
    cap  = cv2.VideoCapture(str(video_path))
    fps  = cap.get(cv2.CAP_PROP_FPS) or 15
    step = max(1, int(fps / cfg.FRAME_RATE))
    scene_list, face_list = [], []
    idx  = 0

    while len(scene_list) < T:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            scene_list.append(_scene_tf(rgb).numpy())
            face_list.append(_get_face_np(rgb))
        idx += 1
    cap.release()

    dummy_s = np.zeros((3, *cfg.SCENE_SIZE), dtype=np.float32)
    dummy_f = np.zeros((3, *cfg.FACE_SIZE),  dtype=np.float32)
    while len(scene_list) < T: scene_list.append(dummy_s)
    while len(face_list)  < T: face_list.append(dummy_f)
    scene_np = np.stack(scene_list[:T]).astype(np.float16)   # [T,3,224,224]
    face_np  = np.stack(face_list[:T]).astype(np.float16)    # [T,3,112,112]

    audio_np = _load_audio_ffmpeg(video_path)                # [1,64,W]

    np.savez_compressed(cache_path,
                        scene=scene_np, face=face_np,
                        audio=audio_np,
                        label=np.array(label, dtype=np.int64))
    return "ok"


def build_cache(label_dict: dict, split_name: str):
    args    = [(s, lbl, p) for s, (lbl, p) in label_dict.items()]
    total   = len(args)
    already = sum(1 for s, _, _ in args
                  if (cfg.CACHE_DIR / f"{s}.npz").exists())

    print(f"\n[Cache:{split_name}] {total:,} clips total  |  "
          f"{already:,} already cached  |  {total - already:,} to build")

    if already == total:
        print(f"[Cache:{split_name}] All clips cached — nothing to do.")
        return

    t0 = time.time()
    done, skipped, errors = 0, 0, 0

    with ThreadPoolExecutor(max_workers=cfg.CACHE_WORKERS) as pool:
        futures = {pool.submit(_cache_clip, a): a[0] for a in args}
        processed = 0
        for fut in as_completed(futures):
            stem = futures[fut]
            try:
                status = fut.result()
                if status == "skip": skipped += 1
                else:                done    += 1
            except Exception as e:
                errors += 1
                print(f"  [WARN] '{stem}': {e}")   # only errors, not every clip

            processed += 1
            if processed % cfg.LOG_EVERY == 0 or processed == total:
                log_progress(processed, total, t0, f"Cache:{split_name}")

    cache_mb = sum(f.stat().st_size for f in cfg.CACHE_DIR.glob("*.npz")) / 1e6
    print(f"[Cache:{split_name}] Done  ->  "
          f"{done} new  |  {skipped} skipped  |  {errors} errors  |  "
          f"Total cache: {cache_mb:.0f} MB")
    gc.collect()

In [ ]:
class DAiSEEDataset(Dataset):
    def __init__(self, label_dict: dict, split_name: str = ""):
        self.items = [
            (stem, label)
            for stem, (label, _) in label_dict.items()
            if (cfg.CACHE_DIR / f"{stem}.npz").exists()
        ]
        missing = len(label_dict) - len(self.items)
        print(f"[Dataset:{split_name}] {len(self.items):,} samples ready"
              + (f"  |  WARNING: {missing} missing cache files" if missing else ""))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        stem, label = self.items[i]
        data = np.load(cfg.CACHE_DIR / f"{stem}.npz",
                       mmap_mode='r', allow_pickle=False)
        return {
            "scene": torch.from_numpy(data["scene"].astype(np.float32)),  # [T,3,224,224]
            "face":  torch.from_numpy(data["face"].astype(np.float32)),   # [T,3,112,112]
            "audio": torch.from_numpy(data["audio"].astype(np.float32)),  # [1,64,W]
            "label": torch.tensor(label, dtype=torch.long),
            "clip":  stem
        }



In [ ]:
class EfficientNetEncoder(nn.Module):
    def __init__(self, d: int = cfg.EMBED_DIM):
        super().__init__()
        bb = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_f = bb.classifier[1].in_features
        bb.classifier = nn.Identity()
        self.backbone = bb
        self.proj = nn.Sequential(nn.Linear(in_f, d), nn.LayerNorm(d))

    def forward(self, x):         
        return self.proj(self.backbone(x))  


class AudioCNNEncoder(nn.Module):
    def __init__(self, d: int = cfg.EMBED_DIM):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.proj = nn.Sequential(nn.Linear(128 * 4 * 4, d), nn.LayerNorm(d))

    def forward(self, x):          # [B, 1, N_MELS, W]
        return self.proj(self.cnn(x).flatten(1))   # [B, D]


In [ ]:
class TransformerTemporalEncoder(nn.Module):
    def __init__(self, d: int = cfg.EMBED_DIM):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=cfg.NHEAD, dim_feedforward=d * 4,
            dropout=cfg.DROPOUT, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=cfg.NUM_LAYERS)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.trunc_normal_(self.cls, std=0.02)

    def forward(self, x):          # [B, T, D]
        cls = self.cls.expand(x.size(0), -1, -1)
        return self.enc(torch.cat([cls, x], dim=1))[:, 0] 


class BiLSTMTemporalEncoder(nn.Module):
    def __init__(self, d: int = cfg.EMBED_DIM):
        super().__init__()
        self.lstm = nn.LSTM(d, d // 2, num_layers=cfg.NUM_LAYERS,
                            batch_first=True, bidirectional=True,
                            dropout=cfg.DROPOUT if cfg.NUM_LAYERS > 1 else 0)

    def forward(self, x):          # [B, T, D]
        out, _ = self.lstm(x)
        return out.mean(dim=1)     # [B, D]



In [ ]:

class AttentionFusion(nn.Module):
    def __init__(self, d: int = cfg.EMBED_DIM):
        super().__init__()
        self.score = nn.Linear(d, 1, bias=False)

    def forward(self, streams: list):   # list of [B, D]
        s = torch.stack(streams, dim=1)                    # [B, n, D]
        w = F.softmax(self.score(s), dim=1)                # [B, n, 1]
        return (w * s).sum(dim=1)                          # [B, D]



In [ ]:

class DAiSEEModel(nn.Module):
    def __init__(self, temporal: str = "transformer"):
        super().__init__()
        D = cfg.EMBED_DIM
        self.face_enc       = EfficientNetEncoder(D)
        self.scene_enc      = EfficientNetEncoder(D)
        self.audio_enc      = AudioCNNEncoder(D)
        Enc = TransformerTemporalEncoder if temporal == "transformer" \
              else BiLSTMTemporalEncoder
        self.face_temporal  = Enc(D)
        self.scene_temporal = Enc(D)
        self.fusion         = AttentionFusion(D)
        self.head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, D // 2), nn.GELU(), nn.Dropout(cfg.DROPOUT),
            nn.Linear(D // 2, cfg.NUM_CLASSES)
        )
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"[Model] Temporal={temporal}  |  Trainable params={total:,}")

    def forward(self,scene,face,audio):
        # scene: [B, T, 3, 224, 224]  face: [B, T, 3, 112, 112]  audio: [B, 1, 64, W]
        B, T = scene.shape[:2]
        s_emb = self.scene_enc(scene.view(B * T, *scene.shape[2:])).view(B, T, -1)
        f_emb = self.face_enc(face.view(B * T,   *face.shape[2:])).view(B, T, -1)
        fused = self.fusion([
            self.face_temporal(f_emb),    # [B, D]
            self.scene_temporal(s_emb),   # [B, D]
            self.audio_enc(audio)         # [B, D]
        ])
        return self.head(fused)           # [B, NUM_CLASSES]



In [ ]:
def run_epoch(model, loader, criterion,
              optimizer=None, scaler=None, tag=""):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    t0      = time.time()
    n_batch = len(loader)

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for i, batch in enumerate(loader, 1):
            scene = batch["scene"].to(cfg.DEVICE, non_blocking=True)
            face  = batch["face"].to(cfg.DEVICE,  non_blocking=True)
            audio = batch["audio"].to(cfg.DEVICE, non_blocking=True)
            label = batch["label"].to(cfg.DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(cfg.DEVICE == "cuda")):
                logits = model(scene, face, audio)
                loss   = criterion(logits, label)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            bs          = label.size(0)
            total_loss += loss.item() * bs
            correct    += (logits.argmax(1) == label).sum().item()
            total      += bs

            if i % cfg.LOG_EVERY == 0 or i == n_batch:
                print(f"  [{tag}] batch {i:>4}/{n_batch}"
                      f"loss={total_loss/total:.4f}"
                      f"acc={correct/total:.4f}"
                      f"time={time.time()-t0:.0f}s")

    return total_loss / total, correct / total


In [ ]:
def train():
    print("\n" + "=" * 58)
    print("STEP 1 - LABEL DICTS")
    print("=" * 58)
    train_dict = build_label_dict(cfg.TRAIN_LABELS, cfg.TRAIN_DIR)
    val_dict   = build_label_dict(cfg.VAL_LABELS,   cfg.VAL_DIR)

    print("\n" + "=" * 58)
    print("  STEP 2 - DISK CACHE  (run once, reused every epoch)")
    print("=" * 58)
    build_cache(train_dict,"Train")
    build_cache(val_dict,"Val")
    gc.collect()

    print("\n" + "=" * 58)
    print("  STEP 3 - DATALOADERS")
    print("=" * 58)
    train_ds = DAiSEEDataset(train_dict,"Train")
    val_ds   = DAiSEEDataset(val_dict,"Val")
    train_dl = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.DL_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2)
    val_dl = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.DL_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2)
    print(f"[DataLoader] Train: {len(train_dl)} batches"
          f"|  Val: {len(val_dl)} batches")
    print("\n" + "=" * 58)
    print("  STEP 4 - MODEL")
    print("=" * 58)
    model     = DAiSEEModel(temporal="transformer").to(cfg.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg.LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=cfg.EPOCHS)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler(enabled=(cfg.DEVICE == "cuda"))
    print(f"[Optimizer] AdamW  lr={cfg.LR}  wd=1e-4")
    print(f"[Scheduler] CosineAnnealingLR  T_max={cfg.EPOCHS}")
    print(f"[AMP]       {'enabled' if cfg.DEVICE == 'cuda' else 'disabled (CPU)'}")
    print("\n" + "=" * 58)
    print("  STEP 5 — TRAINING")
    print("=" * 58)
    best_val_acc = 0.0

    for epoch in range(1, cfg.EPOCHS + 1):
        t0 = time.time()
        print(f"\n── Epoch {epoch:02d}/{cfg.EPOCHS} " + "-" * 40)

        tr_loss, tr_acc = run_epoch(model, train_dl, criterion,
                                    optimizer, scaler, tag="train")
        vl_loss, vl_acc = run_epoch(model, val_dl, criterion,
                                    tag="val")
        scheduler.step()

        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]
        print(f"\n  SUMMARY  epoch={epoch:02d}"
              f"train[loss={tr_loss:.4f} acc={tr_acc:.4f}]"
              f"val[loss={vl_loss:.4f} acc={vl_acc:.4f}]"
              f"lr={lr_now:.2e}  time={elapsed:.0f}s")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), cfg.SAVE_PATH)
            print(f"New best model saved -> val_acc={vl_acc:.4f}"
                  f"({cfg.SAVE_PATH})")
        else:
            print(f" -- No improvement  (best so far: {best_val_acc:.4f})")

        gc.collect()

    print(f"\n{'=' * 58}")
    print(f"  Training complete.  Best val acc = {best_val_acc:.4f}")
    print(f"{'=' * 58}")
    return model, optimizer, scheduler, best_val_acc


In [ ]:
def evaluate_test():
    print("\n" + "=" * 58)
    print("  TEST EVALUATION")
    print("=" * 58)

    test_dict = build_label_dict(cfg.TEST_LABELS, cfg.TEST_DIR)
    build_cache(test_dict, "Test")
    gc.collect()

    test_ds = DAiSEEDataset(test_dict, "Test")
    test_dl = DataLoader(
        test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.DL_WORKERS, pin_memory=True)

    print(f"\n[Test] Loading weights from {cfg.SAVE_PATH} ...")
    model = DAiSEEModel(temporal="transformer").to(cfg.DEVICE)
    model.load_state_dict(torch.load(cfg.SAVE_PATH, map_location=cfg.DEVICE))
    print("[Test] Weights loaded.")

    criterion           = nn.CrossEntropyLoss()
    test_loss, test_acc = run_epoch(model, test_dl, criterion, tag="test")

    print(f"\n{'=' * 58}")
    print(f"  TEST RESULTS")
    print(f"  Loss : {test_loss:.4f}")
    print(f"  Acc  : {test_acc:.4f}")
    print(f"{'=' * 58}")

In [35]:
model, optimizer, scheduler, best_val_acc = train()


  STEP 1 — LABEL DICTS

[Labels] Reading TrainLabels.csv ...
[Labels] Video files found in Train: 5,482
[Labels] Matched  : 5,358  |  Missing: 0
[Labels] Class dist (label:count): {0: 34, 1: 213, 2: 2617, 3: 2494}

[Labels] Reading ValidationLabels.csv ...
[Labels] Video files found in Validation: 1,720
[Labels] Matched  : 1,429  |  Missing: 0
[Labels] Class dist (label:count): {0: 23, 1: 143, 2: 813, 3: 450}

  STEP 2 — DISK CACHE  (run once, reused every epoch)

[Cache:Train] 5,358 clips total  |  0 already cached  |  5,358 to build
  [Cache:Train]   100/5358  (2%)  elapsed=92s  eta=4841s
  [Cache:Train]   200/5358  (4%)  elapsed=169s  eta=4366s
  [Cache:Train]   300/5358  (6%)  elapsed=252s  eta=4243s
  [Cache:Train]   400/5358  (7%)  elapsed=329s  eta=4080s
  [Cache:Train]   500/5358  (9%)  elapsed=398s  eta=3867s
  [Cache:Train]   600/5358  (11%)  elapsed=468s  eta=3712s
  [Cache:Train]   700/5358  (13%)  elapsed=542s  eta=3608s
  [Cache:Train]   800/5358  (15%)  elapsed=616s  et

[mpeg4 @ 0x7fa1bb5c2080] I cbpy damaged at 16 8
[mpeg4 @ 0x7fa1bb5c2080] Error at MB: 344


  [Cache:Train]  2100/5358  (39%)  elapsed=1592s  eta=2470s
  [Cache:Train]  2200/5358  (41%)  elapsed=1666s  eta=2392s
  [Cache:Train]  2300/5358  (43%)  elapsed=1749s  eta=2325s
  [Cache:Train]  2400/5358  (45%)  elapsed=1828s  eta=2252s
  [Cache:Train]  2500/5358  (47%)  elapsed=1896s  eta=2168s
  [Cache:Train]  2600/5358  (49%)  elapsed=1964s  eta=2083s
  [Cache:Train]  2700/5358  (50%)  elapsed=2034s  eta=2002s
  [Cache:Train]  2800/5358  (52%)  elapsed=2124s  eta=1941s
  [Cache:Train]  2900/5358  (54%)  elapsed=2186s  eta=1853s
  [Cache:Train]  3000/5358  (56%)  elapsed=2259s  eta=1775s
  [Cache:Train]  3100/5358  (58%)  elapsed=2333s  eta=1699s
  [Cache:Train]  3200/5358  (60%)  elapsed=2402s  eta=1620s
  [Cache:Train]  3300/5358  (62%)  elapsed=2475s  eta=1544s
  [Cache:Train]  3400/5358  (63%)  elapsed=2555s  eta=1471s
  [Cache:Train]  3500/5358  (65%)  elapsed=2631s  eta=1397s
  [Cache:Train]  3600/5358  (67%)  elapsed=2703s  eta=1320s
  [Cache:Train]  3700/5358  (69%)  elaps

100%|██████████| 20.5M/20.5M [00:00<00:00, 120MB/s] 


[Model] Temporal=transformer  |  Trainable params=12,483,900
[Optimizer] AdamW  lr=0.0001  wd=1e-4
[Scheduler] CosineAnnealingLR  T_max=20
[AMP]       enabled

  STEP 5 — TRAINING

── Epoch 01/20 ────────────────────────────────────────
  [train] batch  100/670  loss=0.9719  acc=0.4450  time=64s
  [train] batch  200/670  loss=0.9330  acc=0.4869  time=108s
  [train] batch  300/670  loss=0.9075  acc=0.5067  time=151s
  [train] batch  400/670  loss=0.8927  acc=0.5144  time=195s
  [train] batch  500/670  loss=0.8864  acc=0.5240  time=239s
  [train] batch  600/670  loss=0.8817  acc=0.5325  time=282s
  [train] batch  670/670  loss=0.8797  acc=0.5321  time=332s
  [val] batch  100/179  loss=0.9551  acc=0.5925  time=26s
  [val] batch  179/179  loss=0.9913  acc=0.5493  time=52s

  SUMMARY  epoch=01  train[loss=0.8797 acc=0.5321]  val[loss=0.9913 acc=0.5493]  lr=9.94e-05  time=383s
  ✓ New best model saved  →  val_acc=0.5493  (/kaggle/working/best_daisee_model.pt)

── Epoch 02/20 ────────────────

In [36]:
evaluate_test()


  TEST EVALUATION

[Labels] Reading TestLabels.csv ...
[Labels] Video files found in Test: 1,866
[Labels] Matched  : 1,784  |  Missing: 0
[Labels] Class dist (label:count): {0: 4, 1: 84, 2: 882, 3: 814}

[Cache:Test] 1,784 clips total  |  0 already cached  |  1,784 to build
  [Cache:Test]   100/1784  (6%)  elapsed=67s  eta=1126s
  [Cache:Test]   200/1784  (11%)  elapsed=136s  eta=1079s
  [Cache:Test]   300/1784  (17%)  elapsed=206s  eta=1019s
  [Cache:Test]   400/1784  (22%)  elapsed=273s  eta=943s
  [Cache:Test]   500/1784  (28%)  elapsed=337s  eta=867s
  [Cache:Test]   600/1784  (34%)  elapsed=479s  eta=945s
  [Cache:Test]   700/1784  (39%)  elapsed=546s  eta=846s
  [Cache:Test]   800/1784  (45%)  elapsed=608s  eta=748s
  [Cache:Test]   900/1784  (50%)  elapsed=670s  eta=658s
  [Cache:Test]  1000/1784  (56%)  elapsed=737s  eta=577s
  [Cache:Test]  1100/1784  (62%)  elapsed=800s  eta=497s
  [Cache:Test]  1200/1784  (67%)  elapsed=868s  eta=422s
  [Cache:Test]  1300/1784  (73%)  elaps

In [ ]:
import shutil, json
from pathlib import Path

EXPORT_DIR = Path("/kaggle/working/daisee_export")
EXPORT_DIR.mkdir(exist_ok=True)

shutil.copy(cfg.SAVE_PATH, EXPORT_DIR / "best_daisee_model.pt")
print(f"[Save] Model weights  ->  {EXPORT_DIR}/best_daisee_model.pt")

torch.save({
    "model_state_dict":     model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_val_acc":         best_val_acc,
    "epoch":                cfg.EPOCHS,
    "config": {
        "NUM_CLASSES": cfg.NUM_CLASSES,
        "TARGET_COL":  cfg.TARGET_COL,
        "EMBED_DIM":   cfg.EMBED_DIM,
        "MAX_FRAMES":  cfg.MAX_FRAMES,
        "FRAME_RATE":  cfg.FRAME_RATE,
        "N_MELS":      cfg.N_MELS,
        "NHEAD":       cfg.NHEAD,
        "NUM_LAYERS":  cfg.NUM_LAYERS,
        "DROPOUT":     cfg.DROPOUT,
    }
}, EXPORT_DIR / "full_checkpoint.pt")
print(f"[Save] Full checkpoint ->  {EXPORT_DIR}/full_checkpoint.pt")

config_dict = {k: str(v) if isinstance(v, Path) else v
               for k, v in cfg.__class__.__dict__.items()
               if not k.startswith("_")}
with open(EXPORT_DIR / "config.json", "w") as f:
    json.dump(config_dict, f, indent=2)
print(f"[Save] Config JSON  ->  {EXPORT_DIR}/config.json")

print("\n[Save] All files:")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"       {f.name:35s}  {f.stat().st_size/1e6:.2f} MB")

[Save] Model weights  →  /kaggle/working/daisee_export/best_daisee_model.pt
[Save] Full checkpoint →  /kaggle/working/daisee_export/full_checkpoint.pt
[Save] Config JSON    →  /kaggle/working/daisee_export/config.json

[Save] All files:
       best_daisee_model.pt                 50.60 MB
       config.json                          0.00 MB
       full_checkpoint.pt                   150.91 MB
